In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [20]:
#@title Input Parameters

batch_size = 64
learning_rate = 0.01
epochs = 3
mnist_training_dataset_mean = 0.1307
mnist_training_dataset_std = 0.3081
mnist_flatten_shape = 784

In [3]:
# Normalize dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((mnist_training_dataset_mean,), (mnist_training_dataset_std,))
])

In [21]:
#@title Download MNIST Dataset
training_data = DataLoader(
    datasets.MNIST('./data', train=True, download=True, transform=transform),
    batch_size=batch_size, shuffle=True
)

test_data = DataLoader(
    datasets.MNIST('./data', train=False, download=True, transform=transform),
    batch_size=1000, shuffle=False
)

In [34]:
class ModelStore:
  def __init__(self):
    self.mlp_architerture = model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(mnist_flatten_shape, 10))
    self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


    self.cnn_architerture = nn.Sequential(
    # Block 1: Capture basic edges
    nn.Conv2d(1, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32), # Normalizes the activations to speed up training
    nn.ReLU(),
    nn.MaxPool2d(2),    # Output: (32, 14, 14)

    # Block 2: Capture complex shapes
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2),    # Output: (64, 7, 7)

    # Classifier
    nn.Flatten(),
    nn.Linear(64 * 7 * 7, 128),
    nn.ReLU(),
    nn.Dropout(0.5),    # Randomly shuts off neurons to prevent overfitting
    nn.Linear(128, 10)
).to(self.device)

  def get_mlp_loss(self):
    return nn.CrossEntropyLoss(),  optim.SGD(self.mlp_architerture.parameters(), lr=learning_rate)

  def get_cnn_loss(self):
    return nn.CrossEntropyLoss(), optim.Adam(self.cnn_architerture.parameters(), lr=0.001)

  def eval_mlp(self, loader):
    self.mlp_architerture.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in loader:
            outputs = self.mlp_architerture(data)

            _, predicted = torch.max(outputs.data, 1)

            total += target.size(0)
            correct += (predicted == target).sum().item()

    accuracy = 100 * correct / total
    print(f'\nAccuracy on the 10,000 test images: {accuracy:.2f}%')

  def eval_cnn(self, loader):
    self.cnn_architerture.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(self.device), target.to(self.device)
            output = self.cnn_architerture(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    accuracy = 100. * correct / len(loader.dataset)
    print(f"\nTest Accuracy: {accuracy:.2f}%")

### Initialize MLP Model & Training


In [38]:
model_store = ModelStore()
mlp_model = model_store.mlp_architerture

# Loss
criterion, optimizer = model_store.get_mlp_loss()

In [39]:
print("Starting Training...")
for epoch in range(epochs):
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(training_data):
        # Forward pass
        output = mlp_model(data)
        loss = criterion(output, target)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if batch_idx % 200 == 199:    # Print every 200 batches
            print(f'Epoch {epoch + 1}, Batch {batch_idx + 1}: Loss {running_loss / 200:.3f}')
            running_loss = 0.0

print("Training Complete!")

Starting Training...
Epoch 1, Batch 200: Loss 0.747
Epoch 1, Batch 400: Loss 0.453
Epoch 1, Batch 600: Loss 0.395
Epoch 1, Batch 800: Loss 0.367
Epoch 2, Batch 200: Loss 0.336
Epoch 2, Batch 400: Loss 0.341
Epoch 2, Batch 600: Loss 0.329
Epoch 2, Batch 800: Loss 0.324
Epoch 3, Batch 200: Loss 0.320
Epoch 3, Batch 400: Loss 0.310
Epoch 3, Batch 600: Loss 0.304
Epoch 3, Batch 800: Loss 0.304
Training Complete!


### Run MLP **Eval** on Test Dataset

In [10]:
model_store.eval_mlp(test_data)


Accuracy on the 10,000 test images: 91.68%


### Initialize CNN Model & Train it

In [35]:
model_store_cnn = ModelStore()
cnn_model = model_store.cnn_architerture

# Loss
cnn_criterion, cnn_optimizer = model_store_cnn.get_cnn_loss()

In [ ]:
print("Starting CNN Training...")
for epoch in range(epochs):
    running_loss = 0.0
    cnn_model.train()
    for batch_idx, (data, target) in enumerate(training_data):
        # Forward pass
        output = cnn_model(data)
        loss = cnn_criterion(output, target)

        # Backward pass and optimization
        cnn_optimizer.zero_grad()
        loss.backward()
        cnn_optimizer.step()

        running_loss += loss.item()
        if batch_idx % 200 == 199:    # Print every 200 batches
            print(f'Epoch {epoch + 1}, Batch {batch_idx + 1}: Loss {running_loss / 200:.3f}')
            running_loss = 0.0

print("Training Complete!")

In [ ]:
evaluate(cnn_model, test_data, model_store.device)

### Run CNN **Eval** on Test Dataset

### **Define** all attacks

In [37]:
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class AdversarialAttacks:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.test_loader = DataLoader(
            datasets.MNIST('./data', train=False, download=True,
                           transform=transforms.ToTensor()),
            batch_size=1, shuffle=True)

    def fgsm_attack(self, image, epsilon, data_grad):
        # Single-step attack
        sign_data_grad = data_grad.sign()
        perturbed_image = image + epsilon * sign_data_grad
        return torch.clamp(perturbed_image, 0, 1)

    def ifgsm_attack(self, model, image, target, epsilon, alpha, iters):
        # Basic Iterative Method (BIM)
        perturbed_image = image.clone().detach().to(self.device)

        for i in range(iters):
            perturbed_image.requires_grad = True
            output = model(perturbed_image)
            loss = F.nll_loss(output, target)

            model.zero_grad()
            loss.backward()

            data_grad = perturbed_image.grad.data
            perturbed_image = perturbed_image + alpha * data_grad.sign()

            # Project back to Epsilon ball
            eta = torch.clamp(perturbed_image - image, min=-epsilon, max=epsilon)
            perturbed_image = torch.clamp(image + eta, min=0, max=1).detach()

        return perturbed_image

    def mifgsm_attack(self, model, image, target, epsilon, alpha, iters, decay=1.0):
        # Momentum Iterative Attack
        perturbed_image = image.clone().detach().to(self.device)
        momentum = torch.zeros_like(image).detach().to(self.device)

        for i in range(iters):
            perturbed_image.requires_grad = True
            output = model(perturbed_image)
            loss = F.nll_loss(output, target)

            model.zero_grad()
            loss.backward()

            grad = perturbed_image.grad.data
            grad = grad / torch.mean(torch.abs(grad), dim=(1,2,3), keepdim=True)
            momentum = decay * momentum + grad

            perturbed_image = perturbed_image + alpha * momentum.sign()
            eta = torch.clamp(perturbed_image - image, min=-epsilon, max=epsilon)
            perturbed_image = torch.clamp(image + eta, min=0, max=1).detach()

        return perturbed_image

    def attack(self, model, epsilon, attack_name, alpha=0.01, iters=10):
        model.eval()
        model.to(self.device)
        correct = 0
        total_attacked = 0

        for data, target in self.test_loader:
            data, target = data.to(self.device), target.to(self.device)

            # Get initial prediction
            output = model(data)
            init_pred = output.max(1, keepdim=True)[1]

            if init_pred.item() != target.item():
                continue

            total_attacked += 1

            # Select Attack Logic
            if attack_name == "fgsm":
                data.requires_grad = True
                output = model(data)
                loss = F.nll_loss(output, target)
                model.zero_grad()
                loss.backward()
                perturbed_data = self.fgsm_attack(data, epsilon, data.grad.data)

            elif attack_name == "ifgsm":
                perturbed_data = self.ifgsm_attack(model, data, target, epsilon, alpha, iters)

            elif attack_name == "mifgsm":
                perturbed_data = self.mifgsm_attack(model, data, target, epsilon, alpha, iters)

            # Check final result
            output = model(perturbed_data)
            final_pred = output.max(1, keepdim=True)[1]

            if final_pred.item() == target.item():
                correct += 1

        final_acc = correct / float(total_attacked) if total_attacked > 0 else 0
        print(f"Attack: {attack_name} | Epsilon: {epsilon} | Success Rate: {1 - final_acc:.4f}")
        return final_acc

### **Run** all attacks

In [40]:
attack_obj = AdversarialAttacks()


In [41]:
attack_obj.attack(mlp_model, 0.01, "fgsm")


Attack: fgsm | Epsilon: 0.01 | Success Rate: 0.0561


0.9439353749698577

In [42]:
attack_obj.attack(mlp_model, 0.05, "fgsm")


Attack: fgsm | Epsilon: 0.05 | Success Rate: 0.3457


0.6543284301904991

In [43]:
attack_obj.attack(mlp_model, 0.10, "fgsm")


Attack: fgsm | Epsilon: 0.1 | Success Rate: 0.7070


0.2929828791897757